In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestRegressor

print("Carregando e processando dados...")
caminho_dados = os.path.abspath(os.path.join(os.getcwd(), '..', '00.Dados', '202608_DadosDafiti_Combinados.csv'))
df = pd.read_csv(caminho_dados)

# Pré-processamento
for col in ['Preço Original', 'Preço Pix', 'Preço Parcelado']:
    nova_coluna = f"{col} Numérico"
    df[nova_coluna] = (df[col].astype(str).str.replace('R$', '', regex=False)
                       .str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.strip())
    df[nova_coluna] = pd.to_numeric(df[nova_coluna], errors='coerce')
    
df['Desconto Pix'] = df['Preço Original Numérico'] - df['Preço Pix Numérico']
df['Tags'] = df['Tags'].fillna('')
colunas_tags = df['Tags'].str.get_dummies(sep=',').add_prefix('Tag_')
df = pd.concat([df, colunas_tags], axis=1)
df = pd.get_dummies(df, columns=['Quantidade de Parcelas'], dtype=int)

colunas_selecionadas = df.columns[9:].tolist()
df_modelo = df[colunas_selecionadas].dropna()

X = df_modelo.drop(columns="Desconto Pix")
y = df_modelo["Desconto Pix"]

print("Treinando Modelo de Média (Previsão)...")
modelo_media = RandomForestRegressor(n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=42)
modelo_media.fit(X, y)

print("Treinando Modelo de Variância (Incerteza)...")
residuos = y - modelo_media.predict(X)
modelo_variancia = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42)
modelo_variancia.fit(X, residuos ** 2)

# Empacota tudo que o Streamlit vai precisar em um dicionário
artefatos = {
    "modelo_media": modelo_media,
    "modelo_variancia": modelo_variancia,
    "colunas_X": X.columns.tolist(),
    "q_adapt": 2.1367 
}

# Salva o arquivo no disco
joblib.dump(artefatos, 'artefatos_precificacao.joblib')
print("✅ Modelos salvos com sucesso em 'artefatos_precificacao.joblib'!")

Carregando e processando dados...
Treinando Modelo de Média (Previsão)...
Treinando Modelo de Variância (Incerteza)...
✅ Modelos salvos com sucesso em 'artefatos_precificacao.joblib'!
